<a href="https://colab.research.google.com/github/prksh830/Healthcare/blob/main/AE_XGB_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc

from imblearn.combine import SMOTETomek

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

import xgboost as xgb
import lightgbm as lgb
!pip install catboost
import catboost as cb
!pip install optuna
import optuna

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.7 MB/s eta 0:00:00


In [ ]:
input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation="relu")(input_layer)
encoded = Dense(32, activation="relu")(encoded)

decoded = Dense(64, activation="relu")(encoded)
decoded = Dense(input_dim, activation="sigmoid")(decoded)

autoencoder = Model(input_layer, decoded)
encoder = Model(input_layer, encoded)

autoencoder.compile(optimizer="adam", loss="mse")

autoencoder.fit(
    X_train, X_train,
    epochs=20,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

X_train_enc = encoder.predict(X_train)
X_test_enc = encoder.predict(X_test)

Epoch 1/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.5218 - val_loss: 0.4631
Epoch 2/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.5086 - val_loss: 0.4623
Epoch 3/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.5083 - val_loss: 0.4622
Epoch 4/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.5081 - val_loss: 0.4620
Epoch 5/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.5080 - val_loss: 0.4620
Epoch 6/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.5079 - val_loss: 0.4620
Epoch 7/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.5078 - val_loss: 0.4620
Epoch 8/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.5078 - val_loss: 0.4619
Epoch 9/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.5078 - val_loss: 0.4619
Epoch 10/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.5078 - val_loss: 0.4621
Epoch 11/20
2954/2954 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.5078 - val_loss: 0.4619
Epoch 12/20
2954/2954 ━━

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }

    model = xgb.XGBClassifier(**params, eval_metric='mlogloss')
    model.fit(X_train_enc, y_train)

    preds = model.predict(X_test_enc)
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

best_params = study.best_params

model_proposed = xgb.XGBClassifier(**best_params, eval_metric='mlogloss')
model_proposed.fit(X_train_enc, y_train)

[I 2026-04-09 10:07:21,706] A new study created in memory with name: no-name-56129ff3-2154-495f-a8b6-feb7fe932e44
[I 2026-04-09 10:09:20,785] Trial 0 finished with value: 0.8125500624779725 and parameters: {'n_estimators': 127, 'max_depth': 4, 'learning_rate': 0.23877535635778283, 'subsample': 0.6692991502486874, 'colsample_bytree': 0.6081828027944866}. Best is trial 0 with value: 0.8125500624779725.
[I 2026-04-09 10:11:51,622] Trial 1 finished with value: 0.9347826086956522 and parameters: {'n_estimators': 180, 'max_depth': 7, 'learning_rate': 0.21489641854849334, 'subsample': 0.958703902744647, 'colsample_bytree': 0.996898289879965}. Best is trial 1 with value: 0.9347826086956522.
[I 2026-04-09 10:15:58,246] Trial 2 finished with value: 0.8919932075229887 and parameters: {'n_estimators': 263, 'max_depth': 9, 'learning_rate': 0.04051717898166367, 'subsample': 0.8529221006378869, 'colsample_bytree': 0.7701848984525559}. Best is trial 1 with value: 0.9347826086956522.
[I 2026-04-09 10:1

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7072662386360613, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.26311237097762485,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=298, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
model_opt_xgb = xgb.XGBClassifier(**best_params, eval_metric='mlogloss')
model_opt_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7072662386360613, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.26311237097762485,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=298, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
input_layer_dnn = Input(shape=(X_train.shape[1],))
x = Dense(128, activation='relu')(input_layer_dnn)
x = Dense(64, activation='relu')(x)
feature_layer = Dense(32, activation='relu')(x)

dnn_model = Model(inputs=input_layer_dnn, outputs=feature_layer)

features_train = dnn_model.predict(X_train)
features_test = dnn_model.predict(X_test)

model_lgb = lgb.LGBMClassifier()
model_lgb.fit(features_train, y_train)

26254/26254 ━━━━━━━━━━━━━━━━━━━━ 39s 1ms/step
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.232228 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8160
[LightGBM] [Info] Number of data points in the train set: 840106, number of used features: 32
[LightGBM] [Info] Start training from score -1.386754
[LightGBM] [Info] Start training from score -1.385435
[LightGBM] [Info] Start training from score -1.385835
[LightGBM] [Info] Start training from score -1.387154


LGBMClassifier()

In [9]:
xgb_model = xgb.XGBClassifier(eval_metric='mlogloss')
cat_model = cb.CatBoostClassifier(verbose=0)

estimators = [
    ('xgb', xgb_model),
    ('cat', cat_model)
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
)

stack_model.fit(X_train_enc, y_train)

StackingClassifier(estimators=[('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogloss',
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction...=None,
                                              learning_rate=None, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None,
                                              max_depth=None, max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=None, n_jobs=None,
                                              num_parallel_tree=None, ...)),
                               ('cat', CatBoostClassifier(verbose=0))],
                   final_estimator=LogisticRegression())

In [10]:
# Ablation 1: Without Autoencoder
model_ab1 = xgb.XGBClassifier()
model_ab1.fit(X_train, y_train)

# Ablation 2: Without Optimization
model_ab2 = xgb.XGBClassifier()
model_ab2.fit(X_train_enc, y_train)

# Ablation 3: Smaller latent space
input_small = Input(shape=(input_dim,))
encoded_small = Dense(16, activation="relu")(input_small)
encoder_small = Model(input_small, encoded_small)

X_train_small = encoder_small.predict(X_train)
X_test_small = encoder_small.predict(X_test)

model_ab3 = xgb.XGBClassifier()
model_ab3.fit(X_train_small, y_train)

26254/26254 ━━━━━━━━━━━━━━━━━━━━ 37s 1ms/step
1951/1951 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [11]:
def save_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(filename, format='tiff', dpi=300)
    plt.close()

In [12]:
def evaluate_model(model, X_tr, X_te, name):
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)

    save_confusion_matrix(y_train, y_pred_train,
                          f"{name} Train CM",
                          f"{name}_train_cm.tiff")

    save_confusion_matrix(y_test, y_pred_test,
                          f"{name} Test CM",
                          f"{name}_test_cm.tiff")

    return model.predict_proba(X_te)

In [13]:
probs = {}

probs["Proposed"] = evaluate_model(model_proposed, X_train_enc, X_test_enc, "Proposed")
probs["Opt_XGB"] = evaluate_model(model_opt_xgb, X_train, X_test, "OptXGB")
probs["DNN_LGB"] = evaluate_model(model_lgb, features_train, features_test, "DNN_LGB")
probs["Stacking"] = evaluate_model(stack_model, X_train_enc, X_test_enc, "Stacking")

# Ablation
probs["Abl1_NoAE"] = evaluate_model(model_ab1, X_train, X_test, "Abl1")
probs["Abl2_NoOpt"] = evaluate_model(model_ab2, X_train_enc, X_test_enc, "Abl2")
probs["Abl3_SmallAE"] = evaluate_model(model_ab3, X_train_small, X_test_small, "Abl3")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [14]:
plt.figure(figsize=(8,6))

for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(y_test_bin.ravel(), prob.ravel())
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()

plt.savefig("ROC_Comparison.tiff", format='tiff', dpi=300)
plt.close()

In [15]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# ==============================
# FUNCTION TO COMPUTE METRICS
# ==============================
def compute_metrics(model, X_tr, X_te, name):
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)

    y_prob = model.predict_proba(X_te)

    results = {}

    # Training Accuracy
    results["Train_Accuracy"] = accuracy_score(y_train, y_pred_train)

    # Testing Metrics
    results["Test_Accuracy"] = accuracy_score(y_test, y_pred_test)
    results["Precision"] = precision_score(y_test, y_pred_test, average='weighted')
    results["Recall"] = recall_score(y_test, y_pred_test, average='weighted')
    results["F1_Score"] = f1_score(y_test, y_pred_test, average='weighted')

    # Multi-class AUC
    try:
        results["AUC"] = roc_auc_score(y_test_bin, y_prob, multi_class='ovr')
    except:
        results["AUC"] = np.nan

    return results


# ==============================
# COLLECT RESULTS FOR ALL MODELS
# ==============================
results_dict = {}

# Main models
results_dict["Proposed"] = compute_metrics(model_proposed, X_train_enc, X_test_enc, "Proposed")
results_dict["Opt_XGB"] = compute_metrics(model_opt_xgb, X_train, X_test, "Opt_XGB")
results_dict["DNN_LGB"] = compute_metrics(model_lgb, features_train, features_test, "DNN_LGB")
results_dict["Stacking"] = compute_metrics(stack_model, X_train_enc, X_test_enc, "Stacking")

# Ablation models
results_dict["Abl1_NoAE"] = compute_metrics(model_ab1, X_train, X_test, "Abl1")
results_dict["Abl2_NoOpt"] = compute_metrics(model_ab2, X_train_enc, X_test_enc, "Abl2")
results_dict["Abl3_SmallAE"] = compute_metrics(model_ab3, X_train_small, X_test_small, "Abl3")


# ==============================
# CONVERT TO DATAFRAME
# ==============================
results_df = pd.DataFrame(results_dict).T

# Round for publication
results_df = results_df.round(4)

print("\n=== FULL MODEL COMPARISON ===")
print(results_df)


# ==============================
# SAVE RESULTS
# ==============================
results_df.to_csv("Model_Comparison_Results.csv")


# ==============================
# SEPARATE ABLATION TABLE
# ==============================
ablation_df = results_df.loc[
    ["Proposed", "Abl1_NoAE", "Abl2_NoOpt", "Abl3_SmallAE"]
]

print("\n=== ABLATION STUDY RESULTS ===")
print(ablation_df)

ablation_df.to_csv("Ablation_Study_Results.csv")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== FULL MODEL COMPARISON ===
              Train_Accuracy  Test_Accuracy  Precision  Recall  F1_Score  \
Proposed              0.9994         0.9738     0.9758  0.9738    0.9746   
Opt_XGB               0.9994         0.9959     0.9961  0.9959    0.9959   
DNN_LGB               0.9204         0.7996     0.9432  0.7996    0.8475   
Stacking              0.9732         0.9254     0.9567  0.9254    0.9362   
Abl1_NoAE             0.9977         0.9927     0.9934  0.9927    0.9929   
Abl2_NoOpt            0.9663         0.8915     0.9540  0.8915    0.9125   
Abl3_SmallAE          0.9302         0.8189     0.9421  0.8189    0.8609   

                 AUC  
Proposed      0.9953  
Opt_XGB       1.0000  
DNN_LGB       0.9724  
Stacking      0.9877  
Abl1_NoAE     1.0000  
Abl2_NoOpt    0.9875  
Abl3_SmallAE  0.9702  

=== ABLATION STUDY RESULTS ===
              Train_Accuracy  Test_Accuracy  Precision  Recall  F1_Score  \
Proposed              0.9994         0.9738     0.9758  0.9738    0.